In [2]:
def split_text(text, chunk_size=50, overlap=10):
    # 1. 무한 루프 방지: chunk_size는 overlap보다 커야 함
    if chunk_size <= overlap:
        raise ValueError("chunk_size는 overlap보다 커야 합니다.")
       
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
       
        # 2. 텍스트의 끝까지 모두 잘랐다면 불필요한 반복 종료
        if end >= len(text):
            break
           
        start = end - overlap
       
    return chunks


In [3]:
long_text = (
    "우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. "
    "강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 "
    "수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 "
    "이상을 수강하면 수료증을 신청할 수 있습니다."
)

# [1번 문제]
# chunk_size=50, overlap=10 으로 나누고 청크 개수 출력
chunks = split_text(long_text, chunk_size=50, overlap=10)
print(f"[1번 문제] 생성된 청크 개수: {len(chunks)}개\n")

# [2번 문제]
# 나눠진 청크를 순서대로 출력하여 겹치는 부분(overlap) 확인
print("[2번 문제] 청크별 내용 및 overlap 확인:")
for i, chunk in enumerate(chunks, 1):
    print(f"--- [청크 {i}] (길이: {len(chunk)}자) ---")
    print(chunk)
    
print("\n" + "=" * 50 + "\n")

# [도전 문제]
# chunk_size=80, overlap=0 으로 다시 나누기
chunks_no_overlap = split_text(long_text, chunk_size=80, overlap=0)
print(f"[도전 문제] 생성된 청크 개수: {len(chunks_no_overlap)}개\n")

for i, chunk in enumerate(chunks_no_overlap, 1):
    print(f"--- [도전 문제 청크 {i}] (길이: {len(chunk)}자) ---")
    print(chunk)


[1번 문제] 생성된 청크 개수: 4개

[2번 문제] 청크별 내용 및 overlap 확인:
--- [청크 1] (길이: 50자) ---
우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. 강의 시작 후 7일
--- [청크 2] (길이: 50자) ---
강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 수강한 강의 비율에 따라
--- [청크 3] (길이: 50자) ---
 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 이상을 수강하면 수료증
--- [청크 4] (길이: 23자) ---
을 수강하면 수료증을 신청할 수 있습니다.


[도전 문제] 생성된 청크 개수: 2개

--- [도전 문제 청크 1] (길이: 80자) ---
우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. 강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 수강한
--- [도전 문제 청크 2] (길이: 63자) ---
 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 이상을 수강하면 수료증을 신청할 수 있습니다.


In [4]:
from dotenv import load_dotenv
from openai import OpenAI

# 1. .env 파일의 환경변수 로드
load_dotenv()

# 2. OpenAI 클라이언트 생성
client = OpenAI()
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector = embeddings.embed_query("안녕하세요")
print(len(vector))  # 1536


1536


In [5]:
from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

#청크 처리된 데이터
docs = [
    "우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. ",
    "강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 ",
    "수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 ",
    "이상을 수강하면 수료증을 신청할 수 있습니다."
]

#벡터 스토어에 문서와 임베딩 모델을 이용해서 벡터로 변환해서 저장
vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)
#질문과 유사도가 높은 벡터 데이터를 2개 찾아서 반환해줌
result=vectorstore.similarity_search("환불은 어떻게 해?", k=2)
#출력해보니: result=[Documnet, Document] -> page_content속성에 내용이 저장되어 있음
print(result)
for doc in result:
    print(doc.page_content)

[Document(id='7b32e46a-4ca6-4eb8-99b7-736944824408', metadata={}, page_content='강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 '), Document(id='dffbfd16-d063-43f2-9d34-ea160c7fd85a', metadata={}, page_content='수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 ')]
강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 
수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
]

# [1번 문제]
# OpenAIEmbeddings()로 embeddings를 만들고, InMemoryVectorStore.from_texts로 docs를 저장한 vectorstore 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)

# [2번 문제]
# vectorstore.similarity_search("환불 받고 싶어요", k=2) 결과를 출력
result = vectorstore.similarity_search("환불 받고 싶어요", k=2)

print(f"[2번 문제] 검색 결과 (k=2, 개수: {len(result)}개):")
for doc in result:
    print(f"- {doc.page_content}")
print("\n" + "=" * 50 + "\n")

# [도전 문제]
# k값을 1과 3으로 각각 바꿔서 검색해보고 결과 문서 개수와 내용 확인

# 1) k=1 인 경우
result_k1 = vectorstore.similarity_search("환불 받고 싶어요", k=1)
print(f"[도전 문제] k=1 검색 결과 (개수: {len(result_k1)}개):")
for doc in result_k1:
    print(f"- {doc.page_content}")

print()

# 2) k=3 인 경우
result_k3 = vectorstore.similarity_search("환불 받고 싶어요", k=3)
print(f"[도전 문제] k=3 검색 결과 (개수: {len(result_k3)}개):")
for doc in result_k3:
    print(f"- {doc.page_content}")


[2번 문제] 검색 결과 (k=2, 개수: 2개):
- 환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
- 수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.


[도전 문제] k=1 검색 결과 (개수: 1개):
- 환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.

[도전 문제] k=3 검색 결과 (개수: 3개):
- 환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
- 수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.
- 수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.


In [7]:
#vectorstore를 langchain에서 사용할 수 있게

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

docs = retriever.invoke("환불 규정이 궁금해요")
for d in docs:
    print(d.page_content)


환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.


In [8]:
# 1. 필수 모듈 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. LLM 모델 생성 (정의되지 않았다면 필수!)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. 검색된 문서들을 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

# 4. 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 답하세요. 모르면 모른다고 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

# 5. RAG 체인 구성 (LCEL)
rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# 6. 질문 실행해보기
answer = rag_chain.invoke("환불 규정이 어떻게 되나요?")
print(answer)


환불 정책에 따르면, 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.


In [10]:
# [1번 문제] retriever 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# [2번 문제] 수료증 질문 검색
print("=== [2번 문제: 수료증 질문] ===")
docs = retriever.invoke("수료증은 언제 받을 수 있나요?")
for d in docs:
    print("-", d.page_content)

print("\n" + "=" * 40 + "\n")

# [도전 문제] 전혀 관련 없는 질문 검색
print("=== [도전 문제: 날씨 질문] ===")
docs_weather = retriever.invoke("오늘 날씨 어때?")
for d in docs_weather:
    print("-", d.page_content)


=== [2번 문제: 수료증 질문] ===
- 수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.
- 수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.


=== [도전 문제: 날씨 질문] ===
- 환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
- 수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.


In [12]:
# 1. 필요한 모듈 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. LLM 모델 준비 (온도 0으로 설정하여 사실 기반 답변 유도)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# [1번 문제] 문서들을 하나의 문자열로 합쳐주는 함수
def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

# [2번 문제] Prompt 및 RAG Chain 구성
prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 답하세요. 문서에 없으면 모른다고 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, 
     "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

# [도전 문제] 
print("=== [1] 문서에 있는 질문: 환불 규정 ===")
answer = rag_chain.invoke("환불 규정이 어떻게 되나요?")
print(answer)

print("\n" + "=" * 40 + "\n")

print("=== [2] 문서에 없는 질문: 날씨 ===")
answer2 = rag_chain.invoke("오늘 날씨가 어때?")
print(answer2)


=== [1] 문서에 있는 질문: 환불 규정 ===
강의 시작 후 7일 이내에는 전액 환불이 가능합니다.


=== [2] 문서에 없는 질문: 날씨 ===
모릅니다.


In [ ]:
# ==============================================================================
# [STEP 0] 필수 라이브러리 import 및 환경설정
# ==============================================================================
import os
from dotenv import load_dotenv

# OpenAI 및 임베딩 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# 메모리 기반 벡터 저장소
from langchain_core.vectorstores import InMemoryVectorStore
# 줄바꿈 및 공백 기준으로 자연스럽게 자르는 최적의 텍스트 분할기
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 프롬프트, 체인 조립 부품들
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()


# ==============================================================================
# [STEP 1] 긴 텍스트 문서(.txt) 생성 (실습용 샘플 파일)
# ==============================================================================
policy_text = """[스마트 아카데미 온라인 강의 운영 및 환불 규정]

제1장 총칙 및 수강 기간
본 규정은 스마트 아카데미가 제공하는 온라인 강의 서비스를 이용하는 모든 수강생에게 적용됩니다.
강의의 기본 수강 기간은 결제 승인일로부터 180일간 유지되며, 기간 만료 7일 전까지 1회에 한하여 30일 무료 연장을 신청할 수 있습니다.

제2장 환불 및 결제 취소 정책
1. 전액 환불: 강의 결제일로부터 7일 이내이고, 전체 강의 중 3강 미만을 수강한 경우 전액 환불됩니다.
2. 부분 환불: 결제일로부터 7일이 경과하였거나 3강 이상 수강한 경우, 수강한 강의 비율과 위약금 10%를 차감한 잔여 금액을 환불합니다.
3. 환불 불가: 전체 수강 기간의 50%가 초과되었거나 진도율이 50%를 넘은 경우에는 환불이 불가능합니다.

제3장 수료증 발급 기준
수료증은 전체 정규 강의 진도율이 80% 이상이고, 최종 실습 과제를 제출하여 100점 만점에 70점 이상을 획득한 수강생에게 전자 PDF 형태로 즉시 발급됩니다.

제4장 고객센터 및 문의 안내
고객센터 운영 시간은 평일 오전 9시부터 오후 6시까지이며, 주말 및 공휴일은 휴무입니다.
문의 사항은 1:1 온라인 게시판 또는 support@smartacademy.com으로 접수해 주시기 바랍니다.
"""

file_path = "lecture_policy.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(policy_text)

print(f"✅ [STEP 1] '{file_path}' 파일이 생성되었습니다.")


# ==============================================================================
# [STEP 2] txt 파일 읽어오기 (Load)
# ==============================================================================
with open(file_path, "r", encoding="utf-8") as f:
    raw_document = f.read()

print(f"✅ [STEP 2] 파일 읽기 완료 (총 글자 수: {len(raw_document)}자)")


# ==============================================================================
# [STEP 3] 텍스트 분할기 설정 및 청킹 (Split / Chunking)
# ==============================================================================
# RecursiveCharacterTextSplitter는 문맥을 최대한 살리기 위해
# [문단 구분(\n\n) -> 줄바꿈(\n) -> 띄어쓰기(' ') -> 글자 단위] 순으로 우선 분할을 시도합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,                         # 각 청크의 최대 글자 수
    chunk_overlap=30,                       # 청크 간 겹치는 글자 수 (문맥 보존용)
    separators=["\n\n", "\n", " ", ""]      # 분할 기준 우선순위
)

chunks = text_splitter.split_text(raw_document)

print(f"✅ [STEP 3] 텍스트 청킹 완료: 총 {len(chunks)}개의 청크 생성")
for i, chunk in enumerate(chunks, 1):
    preview = chunk.replace('\n', ' ')[:40]
    print(f"   - 청크 {i} ({len(chunk)}자): {preview}...")


# ==============================================================================
# [STEP 4] 임베딩 모델 준비 & 벡터 스토어(VectorStore) 생성
# ==============================================================================
# 글자를 의미 기반의 숫자 벡터로 바꿔주는 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 잘라둔 청크들을 벡터로 변환하여 메모리 저장소에 적재
vectorstore = InMemoryVectorStore.from_texts(
    texts=chunks, 
    embedding=embeddings
)
print("✅ [STEP 4] 벡터 저장소(VectorStore) 적재 완료")


# ==============================================================================
# [STEP 5] 검색기(Retriever) 변환 (사서 고용)
# ==============================================================================
# 질문과 가장 연관성이 높은 청크 k=2개를 가져오도록 설정
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)
print("✅ [STEP 5] 검색기(Retriever) 준비 완료 (k=2)")


# ==============================================================================
# [STEP 6] 프롬프트 템플릿 & RAG 체인(LCEL) 조립
# ==============================================================================
# 검색된 청크(문서) 리스트를 줄바꿈 2개로 엮어서 하나의 긴 context 문자열로 합침
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 할루시네이션(거짓말)을 막는 핵심 시스템 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 스마트 아카데미의 친절한 상담원입니다.\n"
        "반드시 제공된 [참고 문서]의 내용에만 근거하여 답변하세요.\n"
        "문서에 없는 내용이거나 알 수 없는 질문이면 절대 지어내지 말고 "
        "'제공된 안내 규정에서는 해당 내용을 확인할 수 없습니다.'라고 정직하게 답하세요.\n\n"
        "[참고 문서]:\n{context}"
    )),
    ("human", "{question}"),
])

# temperature=0으로 설정하여 창작을 배제하고 사실 기반으로만 말하도록 함
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 컨베이어 벨트(체인) 조립:
# [입력 질문] -> [문서검색 & 포맷팅 + 원본질문 패스] -> [프롬프트 완성] -> [LLM 추론] -> [문자열 출력]
rag_chain = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm 
    | StrOutputParser()
)
print("✅ [STEP 6] RAG 체인 파이프라인 조립 완료!\n")


# ==============================================================================
# [STEP 7] 질문 테스트 (정상 질문 vs 할루시네이션 방지 테스트)
# ==============================================================================
print("=" * 60)
print("🔍 [테스트 1] 문서에 명시된 질문: '수료증 발급 기준'")
print("=" * 60)
q1 = "수료증을 받으려면 어떤 조건이 필요한가요?"
print(f"Q: {q1}")
print("A:", rag_chain.invoke(q1))
print("\n")

print("=" * 60)
print("🔍 [테스트 2] 문서에 명시된 질문: '환불 규정'")
print("=" * 60)
q2 = "결제하고 5일 지났는데 전액 환불받을 수 있나요?"
print(f"Q: {q2}")
print("A:", rag_chain.invoke(q2))
print("\n")

print("=" * 60)
print("🛡️ [테스트 3] 문서에 없는 엉뚱한 질문 (할루시네이션 방어 검증)")
print("=" * 60)
q3 = "오늘 저녁 메뉴로 삼겹살 어때?"
print(f"Q: {q3}")
print("A:", rag_chain.invoke(q3))
print("=" * 60)
